In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

 kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

# ============================================================
# KAGGLE DATA ANALYSIS & VISUALIZATION
# Works with Starbucks, Wine Reviews, and most CSV datasets
# ============================================================

import os
import glob
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")

# Chart style
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

print("=" * 60)
print("       KAGGLE DATA ANALYSIS DASHBOARD")
print("=" * 60)

# ------------------------------------------------------------
# 1. FIND DATA FILES
# ------------------------------------------------------------

data_dir = "/kaggle/input"

csv_files = glob.glob(data_dir + "/**/*.csv", recursive=True)
excel_files = (
    glob.glob(data_dir + "/**/*.xlsx", recursive=True)
    + glob.glob(data_dir + "/**/*.xls", recursive=True)
)

all_files = csv_files + excel_files

if not all_files:
    print("\nNo CSV or Excel files found in /kaggle/input.")
    print("Click Add Input in Kaggle and attach your dataset.")
else:
    print("\nAvailable data files:")
    for i, file in enumerate(all_files):
        print(f"{i + 1}. {file}")

    # --------------------------------------------------------
    # 2. LOAD THE FIRST AVAILABLE DATA FILE
    # --------------------------------------------------------

    file_path = all_files[0]

    try:
        if file_path.lower().endswith(".csv"):
            try:
                df = pd.read_csv(file_path)
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding="latin1")
        else:
            df = pd.read_excel(file_path)

        print("\nLoaded file:", file_path)
        print("Dataset shape:", df.shape)

        # ----------------------------------------------------
        # 3. CLEAN COLUMN NAMES
        # ----------------------------------------------------

        df.columns = [
            str(col).strip() for col in df.columns
        ]

        print("\nColumn names:")
        print(df.columns.tolist())

        print("\nFirst 5 rows:")
        display(df.head())

        # ----------------------------------------------------
        # 4. DATA INFORMATION
        # ----------------------------------------------------

        print("\nDataset information:")
        print("Rows:", df.shape[0])
        print("Columns:", df.shape[1])
        print("Duplicate rows:", df.duplicated().sum())

        print("\nMissing values:")
        display(
            df.isnull().sum()
            .sort_values(ascending=False)
            .to_frame("Missing Values")
        )

        print("\nStatistical summary:")
        display(df.describe(include="all").T)

        # ----------------------------------------------------
        # 5. PREPARE COLUMN TYPES
        # ----------------------------------------------------

        numeric_cols = df.select_dtypes(
            include=np.number
        ).columns.tolist()

        categorical_cols = df.select_dtypes(
            include=["object", "category", "bool"]
        ).columns.tolist()

        print("\nNumeric columns:", numeric_cols)
        print("Categorical columns:", categorical_cols)

        # ----------------------------------------------------
        # 6. MISSING VALUES CHART
        # ----------------------------------------------------

        missing = df.isnull().sum()
        missing = missing[missing > 0].sort_values(
            ascending=False
        )

        if len(missing) > 0:
            plt.figure(figsize=(11, 5))
            sns.barplot(
                x=missing.index,
                y=missing.values
            )
            plt.title("Missing Values by Column")
            plt.xlabel("Column")
            plt.ylabel("Missing Values")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()
        else:
            print("\nNo missing values found.")

        # ----------------------------------------------------
        # 7. NUMERIC DISTRIBUTIONS
        # ----------------------------------------------------

        if numeric_cols:
            selected_numeric = numeric_cols[:6]

            df[selected_numeric].hist(
                bins=20,
                figsize=(14, 9),
                edgecolor="black"
            )

            plt.suptitle(
                "Distribution of Numeric Features",
                fontsize=16
            )
            plt.tight_layout()
            plt.show()

        # ----------------------------------------------------
        # 8. CORRELATION HEATMAP
        # ----------------------------------------------------

        if len(numeric_cols) >= 2:
            plt.figure(figsize=(11, 7))

            corr = df[numeric_cols].corr()

            sns.heatmap(
                corr,
                annot=True,
                cmap="coolwarm",
                fmt=".2f",
                linewidths=0.5
            )

            plt.title("Feature Correlation Heatmap")
            plt.tight_layout()
            plt.show()

        # ----------------------------------------------------
        # 9. CATEGORICAL DATA CHARTS
        # ----------------------------------------------------

        for col in categorical_cols[:3]:
            counts = (
                df[col]
                .fillna("Missing")
                .astype(str)
                .value_counts()
                .head(10)
            )

            if len(counts) > 0:
                plt.figure(figsize=(11, 5))

                sns.barplot(
                    x=counts.values,
                    y=counts.index
                )

                plt.title(f"Top Categories: {col}")
                plt.xlabel("Count")
                plt.ylabel(col)
                plt.tight_layout()
                plt.show()

        # ----------------------------------------------------
        # 10. NUMERIC RELATIONSHIP CHART
        # ----------------------------------------------------

        if len(numeric_cols) >= 2:
            x_col = numeric_cols[0]
            y_col = numeric_cols[1]

            plot_df = df[[x_col, y_col]].dropna()

            if len(plot_df) > 0:
                plt.figure(figsize=(10, 6))

                sns.scatterplot(
                    data=plot_df,
                    x=x_col,
                    y=y_col,
                    alpha=0.6
                )

                plt.title(f"{y_col} vs {x_col}")
                plt.tight_layout()
                plt.show()

        # ----------------------------------------------------
        # 11. FINAL SUMMARY
        # ----------------------------------------------------

        print("\n" + "=" * 60)
        print("                 ANALYSIS COMPLETE")
        print("=" * 60)

        print(f"Dataset file: {os.path.basename(file_path)}")
        print(f"Total rows: {df.shape[0]}")
        print(f"Total columns: {df.shape[1]}")
        print(f"Numeric columns: {len(numeric_cols)}")
        print(f"Categorical columns: {len(categorical_cols)}")
        print(f"Missing cells: {df.isnull().sum().sum()}")

    except Exception as error:
        print("\nCould not analyze this file.")
        print("Error:", repr(error))
        print(
            "If multiple datasets are attached, check the "
            "file path and select the correct dataset."
        )


In [ ]:
# =====================================================
# Zenstar024 - Starbucks Coffee Selector
# =====================================================

import pandas as pd
from IPython.display import display

# 1. Starbucks coffee menu (example prices in USD)
coffee_menu = pd.DataFrame([
    {"Coffee": "Caffe Latte", "Category": "Espresso", "Price": 3.75},
    {"Coffee": "Cappuccino", "Category": "Espresso", "Price": 3.75},
    {"Coffee": "Americano", "Category": "Espresso", "Price": 3.25},
    {"Coffee": "Caramel Macchiato", "Category": "Espresso", "Price": 4.75},
    {"Coffee": "Caffe Mocha", "Category": "Espresso", "Price": 4.50},
    {"Coffee": "Cold Brew", "Category": "Cold Coffee", "Price": 3.95},
    {"Coffee": "Iced Latte", "Category": "Cold Coffee", "Price": 4.25},
    {"Coffee": "Caramel Frappuccino", "Category": "Frappuccino", "Price": 4.95},
    {"Coffee": "Chai Tea Latte", "Category": "Tea", "Price": 4.25},
    {"Coffee": "Matcha Tea Latte", "Category": "Tea", "Price": 4.75}
])

print("ZENSTAR024 | STARBUCKS COFFEE MENU")
display(coffee_menu)

# 2. Select coffee category
categories = sorted(coffee_menu["Category"].unique())

print("\nAvailable Categories:")
for i, category in enumerate(categories, 1):
    print(f"{i}. {category}")

while True:
    try:
        category_choice = int(input("\nSelect category number: "))
        if 1 <= category_choice <= len(categories):
            selected_category = categories[category_choice - 1]
            break
        print("Please select a number from the list.")
    except ValueError:
        print("Enter a valid number.")

# 3. Select coffee type
available_coffees = coffee_menu[
    coffee_menu["Category"] == selected_category
].reset_index(drop=True)

print("\nAvailable Coffees:")
for i, row in available_coffees.iterrows():
    print(f"{i + 1}. {row['Coffee']} - ${row['Price']:.2f}")

while True:
    try:
        coffee_choice = int(input("\nSelect coffee number: "))
        if 1 <= coffee_choice <= len(available_coffees):
            coffee = available_coffees.iloc[coffee_choice - 1]
            break
        print("Please select a number from the list.")
    except ValueError:
        print("Enter a valid number.")

# 4. Select size
sizes = {
    "1": ("Short", 0.00),
    "2": ("Tall", 0.50),
    "3": ("Grande", 1.00),
    "4": ("Venti", 1.50)
}

print("\nSelect Size:")
for key, value in sizes.items():
    print(f"{key}. {value[0]}")

while True:
    size_choice = input("Enter size number: ").strip()
    if size_choice in sizes:
        size, extra_price = sizes[size_choice]
        break
    print("Choose 1, 2, 3, or 4.")

# 5. Select milk
milks = {
    "1": "Whole Milk",
    "2": "2% Milk",
    "3": "Nonfat Milk",
    "4": "Oat Milk",
    "5": "Almond Milk",
    "6": "Soy Milk",
    "7": "Coconut Milk",
    "8": "No Milk"
}

print("\nSelect Milk:")
for key, milk in milks.items():
    print(f"{key}. {milk}")

while True:
    milk_choice = input("Enter milk number: ").strip()
    if milk_choice in milks:
        milk = milks[milk_choice]
        break
    print("Choose a number from 1 to 8.")

# 6. Select sugar preference
sugar_options = {
    "1": "No Added Sugar / Syrup",
    "2": "Less Sweet",
    "3": "Regular Sweetness",
    "4": "Extra Sweet"
}

print("\nSelect Sugar Preference:")
for key, value in sugar_options.items():
    print(f"{key}. {value}")

while True:
    sugar_choice = input("Enter sugar option: ").strip()
    if sugar_choice in sugar_options:
        sugar = sugar_options[sugar_choice]
        break
    print("Choose a number from 1 to 4.")

# 7. Enter quantity
while True:
    try:
        quantity = int(input("\nEnter quantity: "))
        if quantity > 0:
            break
        print("Quantity must be at least 1.")
    except ValueError:
        print("Enter a whole number, such as 2.")

# 8. Calculate example total
unit_price = round(float(coffee["Price"]) + extra_price, 2)
total_price = round(unit_price * quantity, 2)

# 9. Display order summary
order = pd.DataFrame([{
    "Coffee": coffee["Coffee"],
    "Category": selected_category,
    "Size": size,
    "Milk": milk,
    "Sugar Preference": sugar,
    "Quantity": quantity,
    "Example Unit Price USD": unit_price,
    "Example Total USD": total_price
}])

print("\n" + "=" * 45)
print("ZENSTAR024 - YOUR COFFEE ORDER")
print("=" * 45)

display(order)

print(f"Total Drinks: {quantity}")
print(f"Estimated Total: ${total_price:.2f}")

# 10. Save results in Kaggle
order.to_csv(
    "/kaggle/working/zenstar024_coffee_order.csv",
    index=False
)

coffee_menu.to_csv(
    "/kaggle/working/zenstar024_coffee_menu.csv",
    index=False
)

print("\nOrder saved successfully!")
print("File: /kaggle/working/zenstar024_coffee_order.csv")